# CreditGuard — Data Preparation & Exploratory Analysis
## Notebook 01: Dataset Ingestion, Cleaning, and Preprocessing Pipeline

This notebook documents the end-to-end data preparation workflow for **CreditGuard**, an educational machine-learning decision-support tool for retail credit default risk prediction.

### Objectives:
1. **Safely Ingest** the official UCI German Credit dataset.
2. **Decode & Map** historical categorical attributes into intuitive domain terms.
3. **Clean & Validate** data integrity (handle duplicates, invalid values, impute missing values).
4. **Perform Exploratory Data Analysis (EDA)** on borrower attributes and class imbalance (70% Non-Default vs 30% Default).
5. **Build Preprocessing Pipelines** using Scikit-Learn `ColumnTransformer`, `StandardScaler`, and `OneHotEncoder` to guarantee zero data leakage.
6. **Split and Save** stratified training and testing cohorts.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import (
    load_raw_data,
    clean_data,
    build_preprocessor,
    split_and_save_data,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COLUMN
)

print("Imports successfully loaded.")


### 1. Ingesting Raw German Credit Data
We load the dataset using `load_raw_data()`, which downloads from the official UCI ML repository if not present locally.


In [ ]:
raw_df = load_raw_data()
print("Raw Dataset Shape:", raw_df.shape)
raw_df.head(5)


### 2. Data Cleaning & Validation
- Check duplicate rows and remove if any.
- Fill any missing numeric features with column median.
- Fill any missing categorical features with `"Unknown"`.
- Ensure binary target representation (0: Non-Default / Good, 1: Default / Bad).


In [ ]:
cleaned_df = clean_data(raw_df)
print("Cleaned shape:", cleaned_df.shape)
print("Missing values per column:\n", cleaned_df.isnull().sum()[cleaned_df.isnull().sum() > 0])
print("\nTarget class breakdown:")
print(cleaned_df[TARGET_COLUMN].value_counts(normalize=True).rename({0: "Non-Default (Good)", 1: "Default (Risky)"}))


### 3. Exploratory Data Analysis (EDA)
Let us inspect the distributions of loan durations, credit amounts, and applicant ages across repayment outcomes.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Loan Duration
cleaned_df.boxplot(column="duration_months", by=TARGET_COLUMN, ax=axes[0])
axes[0].set_title("Loan Duration (Months) by Outcome")
axes[0].set_xlabel("0 = Good Credit, 1 = Default")

# Credit Amount
cleaned_df.boxplot(column="credit_amount", by=TARGET_COLUMN, ax=axes[1])
axes[1].set_title("Credit Amount (DM) by Outcome")
axes[1].set_xlabel("0 = Good Credit, 1 = Default")

# Applicant Age
cleaned_df.boxplot(column="age_years", by=TARGET_COLUMN, ax=axes[2])
axes[2].set_title("Applicant Age (Years) by Outcome")
axes[2].set_xlabel("0 = Good Credit, 1 = Default")

plt.suptitle("")
plt.tight_layout()
plt.show()


### 4. Constructing Scikit-Learn Preprocessing Pipeline
We use `ColumnTransformer` to encapsulate all numeric scaling and categorical one-hot encoding. This structure ensures zero data leakage when fitted solely on the training fold.


In [ ]:
preprocessor = build_preprocessor()
print("Numeric Features:", NUMERIC_FEATURES)
print("Categorical Features:", CATEGORICAL_FEATURES)
print("Preprocessor structure:", preprocessor)


### 5. Stratified Train-Test Split (80% / 20%)
We partition the dataset using `stratify=y` and `random_state=42` to preserve the 70/30 class proportion across both subsets.


In [ ]:
train_df, test_df = split_and_save_data(cleaned_df, test_size=0.20, random_state=42)
print("Training records:", len(train_df), "with default rate:", train_df[TARGET_COLUMN].mean())
print("Testing records: ", len(test_df), "with default rate:", test_df[TARGET_COLUMN].mean())
